# KAT-VAD v2 — **Nexar full read (Amendment 18, §29)**: `nexar_test` once + zero-shot TAD / DADA / DoTA

Rules `core/docs/v2/PREREG_ADDENDUM.md` **§29** (D-N16…D-N22). No training: run `nexar_train.ipynb` to the end
first (25 Nexar `window` checkpoints). Every read is the **ordinary** protocol of its set, no clip-level gate (D-N21).

| set | who | protocol | tool |
|---|---|---|---|
| **`nexar_test`** (sealed, **opened once**) | Nexar-trained × 5 arms; **T2-trained × 5 arms** (printed beside) | N-B: `s1[::3]` per item, crops + whole | `nexar_eval --final` |
| **TAD** test (100 videos) | Nexar-trained × 5 | §26: s8, `core.evaluate --set data.dataset=TAD`, raw | `core.evaluate` |
| **DADA-2000 whole videos** (T2-test's sources) | Nexar-trained × 5 | every s8 row, one forward, annotation span | `dada_whole_eval` |
| **DoTA whole clips** (DoTA-CAP dev + eval, 1,129) | Nexar-trained × 5 | protocol B: `s1[::3]`, whole clip | `protocol_b_eval` |

Nexar-trained A1–A3-ign read every set through inputs baked with the **Nexar** statistics (`build_v2_inputs apply`);
T2-trained arms read `nexar_test` with **their own T2** statistics. Seeds averaged per frame inside each arm.

| step | what | GPU? |
|---|---|---|
| 0 | setup | — |
| 1 | preflight: 50 checkpoints (J10 + config), the six bakes' statistics, every feature cache | no |
| 2 | bake A1 / A2 / A3 inputs with the Nexar statistics on TAD, DADA, DoTA-CAP dev / eval → Drive | no |
| 3 | **`nexar_test`** — marker, then 10 `nexar_eval --final` calls (5 Nexar + 5 T2 arms) | yes |
| 4 | TAD — 25 `core.evaluate` runs | yes |
| 5 | DADA whole — 5 `dada_whole_eval` calls | yes |
| 6 | DoTA — 10 `protocol_b_eval` calls (dev, eval `--final`) | yes |
| 7 | `nexar_transfer_readout` → **paste `transfer_readout.md` to Claude** | no |

Every step skips an output already on Drive (once means once, D-N22). Outputs: `Thesis-V2/outputs/v2_nexar_window/eval/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/{TAD,DoTA,DADA2000_orig,DADA2000Origin}, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read: kat-vad/, cache/, data/, outputs/v2_e3, v2_e4_0; write: cache/v2, outputs/v2_nexar_window
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert (REPO / 'core/tools/nexar_transfer_readout.py').is_file(), 'this upload predates §29 -- re-upload branch v2'
addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 29. Amendment 18' in addendum, 'upload has no addendum §29'
assert 'Authorized by the user on' in addendum.split('## 29. Amendment 18', 1)[1], '§29 is not signed'

SEEDS    = list(constants.NEXAR_RUN_SEEDS)              # 2024-2028 (Nexar, E3 and E4-0 trained the same five)
STEP     = constants.NEXAR_STEP_BUDGET                  # 1,740 for every checkpoint read here (J10)
CRN      = 'R2'
V2S      = constants.VIDEOMAE_ENCODER_S                 # 'vit_s_k710_dl_from_giant'
OFF      = constants.V2_OFF
ARMS     = list(constants.NEXAR_FULL_ARMS)              # A0, A1, A2, A3, A3ign
ANCHOR   = dict(constants.NEXAR_FULL_ANCHOR)            # A0-A3 span, A3ign ignore (both corpora)
EXPECT   = {'A0': (OFF, OFF), 'A1': (CRN, OFF), 'A2': (OFF, V2S), 'A3': (CRN, V2S), 'A3ign': (CRN, V2S)}
BAKE_TAG = {'A1': f'A1_{CRN}', 'A2': 'A2_S', 'A3': f'A3_{CRN}_S'}
BAKE_OF  = {'A1': 'A1', 'A2': 'A2', 'A3': 'A3', 'A3ign': 'A3'}
T2NAME   = constants.DADA_ORIGIN_DATASET                # 'DADA2000_orig'
TAD_NAME = constants.TAD_DATASET
CAP_SPLITS = (constants.V2_SPLIT_DOTA_CAP_DEV, constants.V2_SPLIT_DOTA_CAP_EVAL)

V2_CACHE = Path(DRIVE_V2) / 'cache'
ROOT     = constants.OUTPUT_ROOT / constants.NEXAR_FULL_RUNS_DIR      # v2_nexar_window/
OUT      = ROOT / 'eval'
RUNS     = {'nexar': {a: ROOT / a for a in ARMS},
            't2': {**{a: constants.OUTPUT_ROOT / 'v2_e3' / a for a in ('A0', 'A1', 'A2', 'A3')},
                   'A3ign': constants.OUTPUT_ROOT / 'v2_e4_0' / constants.V2_E4_0_ARM}}
STATS    = {'nexar': {b: V2_CACHE / 'v2' / t / 'Nexar_s8' for b, t in BAKE_TAG.items()},   # nexar_train.ipynb step 3
            't2': {b: V2_CACHE / 'v2' / t / T2NAME for b, t in BAKE_TAG.items()}}          # E3's bakes

# Nexar
CENSUS   = constants.OUTPUT_ROOT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
CLIP_S1  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
VIDEO_S1 = V2_CACHE / 'video' / V2S / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
DATA_N   = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'   # defs.json
# TAD (§26's inputs)
TAD      = constants.DATA_ROOT / TAD_NAME
TAD_NCC  = constants.CLIP_CACHE_DIR / 'TAD_ncc'
TAD_V2S  = V2_CACHE / 'video' / V2S / f'{TAD_NAME}_s{constants.V2_TADZ_STRIDE}_squash'   # written by tad_zeroshot
# DADA-2000 original (T2's sources)
T2       = constants.DATA_ROOT / T2NAME
T2_CLIP  = constants.CLIP_CACHE_DIR / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
# DoTA
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
CAP_S1   = constants.VIDEO_CACHE_DIR / V2S / f'{constants.DOTA_CAP_DATASET}_s1_squash'

BAKED = {  # Nexar statistics applied to each zero-shot set (written in step 2)
    'tad': {b: V2_CACHE / 'v2' / t / f'{TAD_NAME}_s8_nexar' for b, t in BAKE_TAG.items()},
    'dada': {b: V2_CACHE / 'v2' / t / f'{T2NAME}_test_sources_nexar' for b, t in BAKE_TAG.items()},
    **{s: {b: V2_CACHE / 'v2' / t / f'{s}_s3_from_s1_nexar' for b, t in BAKE_TAG.items()} for s in CAP_SPLITS},
}
LOCAL = Path('/content/nexar_full')
LOCAL.mkdir(parents=True, exist_ok=True)
for name in ('ROOT', 'OUT', 'CLIP_S1', 'TAD_V2S', 'T2_CLIP', 'ANNOT', 'DOTA_S1', 'CAP_S1'):
    print(f'{name:9s}', globals()[name])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu openpyxl
python -c "import transformers, openpyxl; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 3-6 will be slow'
df -h /content | tail -1

In [ ]:
import json
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=()):
    """Copy ``ids`` (+ the dir's manifests / stats) from Drive to VM disk once (bit-identical; C2)."""
    dst.mkdir(parents=True, exist_ok=True)
    for v in ids:
        if not (dst / f'{v}.npy').exists():
            shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
    for pattern in ('*.json', '*.npz'):
        for m in src.glob(pattern):
            shutil.copy2(m, dst / m.name)
    return dst


def publish(src, dst):
    """Copy a VM-local result to Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (lesson (bi))."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def ckpt(src, arm, seed):
    return RUNS[src][arm] / f's{seed}' / 'checkpoint_last.pt'


def run_args(src, arm):
    return [x for s in SEEDS for x in ('--run', f's{s}', ckpt(src, arm, s))]


def complete(d, ids):
    return d.is_dir() and all((d / f'{v}.npy').exists() for v in ids)

## 1. Preflight — every checkpoint is finished and is the arm it claims (J10, D-N22)

In [ ]:
from core.data.v2_splits import load_split
from core.tools.dada_whole_eval import test_sources

for src in ('nexar', 't2'):
    for arm in ARMS:
        for s in SEEDS:
            d = RUNS[src][arm] / f's{s}'
            step, last = ckpt_step(d / 'checkpoint_last.pt'), metrics_last(d / 'metrics.jsonl')
            assert last is not None and step == last['global_step'] == STEP, f'{src} {arm} s{s}: J10 (ckpt {step})'
            cfg = yaml.safe_load((d / 'config.yaml').read_text())
            got = (cfg['v2']['crn'], cfg['v2']['motion'], cfg['loss']['dvs_anchor_mode'])
            assert got == (*EXPECT[arm], ANCHOR[arm]), f'{src} {arm} s{s}: config {got}'
print(f'{2 * len(ARMS) * len(SEEDS)} checkpoints OK at step {STEP} (25 Nexar window + 25 T2)')

for src, by_bake in STATS.items():
    sha = set()
    for b, d in by_bake.items():
        m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
        assert (m['crn'], m['motion']) == EXPECT[b], f'{src} {b}: bake {d} is {(m["crn"], m["motion"])}'
        assert (d / 'v2_input_stats.npz').is_file(), f'{src} {b}: no stats in {d}'
        sha.add(m['train_ids_sha1'])
    assert len(sha) == 1, f'{src}: bakes fitted on different splits {sha}'
    print(f'{src} statistics: one train split {sha.pop()[:12]} for A1 / A2 / A3')

TEST = load_split(constants.V2_SPLIT_NEXAR_TEST, final=True)          # ids only; no score is read here
miss = [v for v in TEST if not (CLIP_S1 / f'{v}.npy').exists() or not (VIDEO_S1 / f'{v}.npy').exists()]
assert not miss, f'{len(miss)} nexar_test videos lack N2 features: {miss[:5]}'
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(TAD_IDS) == constants.V2_TADZ_VIDEOS and complete(TAD_NCC, TAD_IDS) and complete(TAD_V2S, TAD_IDS), \
    'TAD inputs incomplete -- run tad_zeroshot.ipynb steps 1-4 first'
DADA_IDS = test_sources(T2)
cands = [constants.VIDEO_CACHE_DIR / V2S / f'{T2NAME}_s8_squash', V2_CACHE / 'video' / V2S / f'{T2NAME}_s8_squash']
T2_VIDEO = next((c for c in cands if complete(c, DADA_IDS)), None)
assert T2_VIDEO is not None, f'no T2 V2-S s8 cache with every DADA test source; looked in {cands}'
assert complete(T2_CLIP, DADA_IDS) and ANNOT.is_file() and T2_CENSUS.is_file(), 'DADA inputs incomplete'
CAP = {s: load_split(s, final=(s == constants.V2_SPLIT_DOTA_CAP_EVAL)) for s in CAP_SPLITS}
CAP_IDS = sorted(CAP[CAP_SPLITS[0]] + CAP[CAP_SPLITS[1]])
assert complete(DOTA_S1, CAP_IDS) and complete(CAP_S1, CAP_IDS), 'DoTA-CAP inputs incomplete'
print(f'nexar_test {len(TEST)} · TAD {len(TAD_IDS)} · DADA sources {len(DADA_IDS)} (V2-S {T2_VIDEO}) · '
      f'DoTA-CAP {len(CAP[CAP_SPLITS[0]])} + {len(CAP[CAP_SPLITS[1]])}')

## 2. Bake A1 / A2 / A3 with the **Nexar** statistics on TAD, DADA, DoTA-CAP (CPU, → Drive)

`build_v2_inputs apply`: CRN referenced per clip (TAD video, DADA source, DoTA clip), the Nexar-fitted `s`, `c`.
TAD and DADA at their s8 rows (stride 1 over the s8 caches); DoTA at `s1[::3]` (protocol B). A3-ign reads A3's bake.

In [ ]:
L = LOCAL / 'stage'
L_TAD_X = stage(TAD_NCC, L / 'TAD_ncc', TAD_IDS)
L_TAD_U = stage(TAD_V2S, L / 'TAD_v2s', TAD_IDS)
L_T2_X = stage(T2_CLIP, L / 'T2_clip', DADA_IDS)
L_T2_U = stage(T2_VIDEO, L / 'T2_v2s', DADA_IDS)
L_DOTA_S1 = stage(DOTA_S1, L / 'DoTA_s1', CAP_IDS)
L_STATS = {src: {b: stage(d, L / f'stats_{src}_{b}') for b, d in by.items()} for src, by in STATS.items()}
SOURCES = {  # set -> (ids, CLIP dir, V2-S dir, stride over those dirs)
    'tad': (TAD_IDS, L_TAD_X, L_TAD_U, 1),
    'dada': (DADA_IDS, L_T2_X, L_T2_U, 1),
    **{s: (CAP[s], L_DOTA_S1, CAP_S1, constants.NEXAR_EVAL_STRIDE) for s in CAP_SPLITS},
}
for name, (ids, x_dir, u_dir, stride) in SOURCES.items():
    ids_file = write_ids(LOCAL / 'ids' / f'{name}.txt', ids)
    for b, dst in BAKED[name].items():
        man = dst / constants.V2_INPUT_MANIFEST_FILENAME
        if man.exists() and json.loads(man.read_text())['baked_ids'] == len(ids):
            continue
        video = ['--video-dir', u_dir] if EXPECT[b][1] != OFF else []
        local = LOCAL / 'bake' / name / b
        run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', L_STATS['nexar'][b],
             '--clip-dir', x_dir, *video, '--ids-file', ids_file, '--stride', stride, '--out-dir', local],
            log=LOCAL / 'bake.log')
        publish(local, dst)
nexar_sha = json.loads((STATS['nexar']['A3'] / constants.V2_INPUT_MANIFEST_FILENAME).read_text())['train_ids_sha1']
for name, by in BAKED.items():
    for b, d in by.items():
        m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
        assert (m['crn'], m['motion']) == EXPECT[b] and m['train_ids_sha1'] == nexar_sha, f'{name} {b}: {m}'
        assert m['baked_ids'] == len(SOURCES[name][0]), f'{name} {b}: baked {m["baked_ids"]}'
print('every zero-shot bake carries the Nexar statistics', nexar_sha[:12])

## 3. `nexar_test` — **opened once** (D-N22): 5 Nexar-trained + 5 T2-trained arms

The marker is written before the first read. An arm whose read-out is on Drive is never re-scored.

In [ ]:
L_TEST_X = stage(CLIP_S1, L / 'nexar_test_clip', TEST)
L_TEST_U = stage(VIDEO_S1, L / 'nexar_test_v2s', TEST)
L_DEFS = stage(DATA_N, L / 'nexar_defs')
marker = OUT / 'nexar_test' / constants.NEXAR_TEST_MARKER
if not marker.exists():
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text(json.dumps({'opened_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                                  'addendum': '§29 Amendment 18', 'seeds': SEEDS,
                                  'arms': {src: ARMS for src in RUNS}}, indent=2))
    print('nexar_test OPENED now:', marker)
else:
    print('resuming -- nexar_test opened on', json.loads(marker.read_text())['opened_utc'])
for src in ('nexar', 't2'):
    for arm in ARMS:
        out = OUT / 'nexar_test' / src / arm
        if (out / 'nexar_readout.json').exists():
            print(f'{src}/{arm}: already read -- not re-scored')
            continue
        stats = [] if arm == 'A0' else ['--stats-dir', L_STATS[src][BAKE_OF[arm]]]
        video = ['--video-s1-dir', L_TEST_U] if EXPECT[arm][1] != OFF else []
        local = LOCAL / 'nexar_test' / src / arm
        t0 = time.time()
        run([sys.executable, '-m', 'core.tools.nexar_eval', *run_args(src, arm), '--census', CENSUS,
             '--clip-s1-dir', L_TEST_X, *stats, *video, '--data-dir', L_DEFS,
             '--split', constants.V2_SPLIT_NEXAR_TEST, '--final', '--out-dir', local],
            log=LOCAL / f'nexar_test_{src}_{arm}.log')
        publish(local, out)
        r = json.loads((out / 'nexar_readout.json').read_text())
        print(f"{src}/{arm}: crop endpoint {r['crops']['mean_over_placements']['macro']['mean']:.4f} "
              f"({(time.time() - t0) / 60:.1f} min)")

## 4. TAD — §26's protocol, Nexar-trained × 5 arms × 5 seeds (`core.evaluate`)

In [ ]:
L_TAD = L / 'TAD'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never the frame archive
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
L_TAD_IN = {'A0': L_TAD_X, **{b: stage(d, L / f'TAD_{b}', TAD_IDS) for b, d in BAKED['tad'].items()}}
for arm in ARMS:
    for s in SEEDS:
        out = OUT / 'tad' / arm / f's{s}'
        if (out / 'results.json').exists():
            continue
        local = LOCAL / 'tad' / arm / f's{s}'
        if local.exists():
            shutil.rmtree(local)
        clip_dir = L_TAD_IN['A0' if arm == 'A0' else BAKE_OF[arm]]
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt('nexar', arm, s),
             '--set', f'data.dataset={TAD_NAME}', '--data-dir', L_TAD, '--clip-dir', clip_dir,
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'tad_{arm}_s{s}.log')
        publish(local, out)
        r = json.loads((out / 'results.json').read_text())
        print(f'tad {arm} s{s}: macro {r["auc_macro"]:.4f} | micro {r["auc"]:.4f}')

## 5. DADA-2000 whole videos — the sources of T2-test, every s8 row (`dada_whole_eval`)

In [ ]:
L_T2_META = stage(T2, L / 'T2_meta')                 # meta.json / defs (json only)
L_DADA_IN = {'A0': L_T2_X, **{b: stage(d, L / f'DADA_{b}', DADA_IDS) for b, d in BAKED['dada'].items()}}
for arm in ARMS:
    out = OUT / 'dada' / arm
    if (out / 'dada_readout.json').exists():
        print(f'dada {arm}: already read')
        continue
    local = LOCAL / 'dada' / arm
    run([sys.executable, '-m', 'core.tools.dada_whole_eval', *run_args('nexar', arm), '--t2-dir', L_T2_META,
         '--annotation', ANNOT, '--census', T2_CENSUS,
         '--input-dir', L_DADA_IN['A0' if arm == 'A0' else BAKE_OF[arm]], '--data-dir', L_DEFS,
         '--out-dir', local], log=LOCAL / f'dada_{arm}.log')
    publish(local, out)
    r = json.loads((out / 'dada_readout.json').read_text())
    print(f"dada {arm}: macro {r['macro']['mean']:.4f} over {r['two_class']} sources | micro {r['micro']:.4f}")

## 6. DoTA whole clips — protocol B on DoTA-CAP dev + eval (`protocol_b_eval`; eval was opened by §19)

In [ ]:
L_DOTA_LABELS = stage(DOTA / 'labels_s8', L / 'dota_labels')
for split in CAP_SPLITS:
    L_CAP_IN = {'A0': L_DOTA_S1, **{b: stage(d, L / f'{split}_{b}', CAP[split]) for b, d in BAKED[split].items()}}
    final = ['--final'] if split == constants.V2_SPLIT_DOTA_CAP_EVAL else []
    for arm in ARMS:
        out = OUT / 'dota' / arm / split
        if (out / 'protocol_b_readout.json').exists():
            print(f'dota {arm} {split}: already read')
            continue
        local = LOCAL / 'dota' / arm / split
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', *run_args('nexar', arm), *final,
             '--input-dir', L_CAP_IN['A0' if arm == 'A0' else BAKE_OF[arm]], '--s1-dir', L_DOTA_S1,
             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', L_DOTA_LABELS, '--split', split, '--out-dir', local, '--device', 'auto'],
            log=LOCAL / f'dota_{arm}_{split}.log')
        publish(local, out)
        r = json.loads((out / 'protocol_b_readout.json').read_text())
        print(f"dota {arm} {split}: macro {r['macro']['mean']:.4f} over {r['clips']} clips")

## 7. Read-out (§29 D-N18 first, then the printed table) → paste `transfer_readout.md` to Claude

In [ ]:
run([sys.executable, '-m', 'core.tools.nexar_transfer_readout', '--root', ROOT])
print((ROOT / 'REPORTS' / 'transfer_readout.md').read_text())